# 🚀 GIAI ĐOẠN 5 — KIẾN TRÚC STAIR5-v4 / NLGCL-CSE: CANDIDATE SUPPORT EXPANSION
### 🏆 Kaggle ML Engineering Pipeline — Multimodal Recommendation with Candidate Support Expansion & Faithful Neighborhood-enriched Contrastive Learning

> **Đề tài:** Recommender Systems using Graph Representation: Multi-modal  
> **Khóa luận tốt nghiệp:** Khóa 2021–2025 — Khoa Công nghệ Thông tin, Trường Đại học Khoa học Tự nhiên, ĐHQG-HCM  
> **Sinh viên thực hiện:** Lê Hà Thanh Chương (23120195) & Bùi Trung Hiếu (23120257)  
> **Giảng viên hướng dẫn:** TS. Nguyễn Ngọc Thảo  
> **Kiến trúc:** **STAIR5-v4 / NLGCL-CSE** (*Neighborhood-enriched Contrastive Learning with Candidate Support Expansion*)  
> **Tài liệu phương pháp luận & Báo cáo:** `docs/giai_doan_5/STAIR5_v4_Report.md`  
> **Mã nguồn lõi:** `models/stair5_v4.py`, `models/stair5_v4_graph.py`, `models/stair5_v4_objectives.py`, `models/stair5_v4_utils.py`, `optimizers/stair5_v4_smoother.py`, `main_stair5_v4.py`  
> **Tập dữ liệu mục tiêu:** **Amazon Sports**, **Amazon Baby**, **Amazon Electronics** (3 tập chuẩn thương mại điện tử)  
> **Chiến lược thực thi:** **3-Tier Pipeline** (`Pha A: Sports [Pilot] ➔ Pha B: Baby [Confirmatory] ➔ Pha C: Electronics [Scale-up]`)  
> **Đo lường VRAM:** Paper Standard (`torch.cuda.max_memory_allocated()`, trích xuất trực tiếp từ `training_telemetry.jsonl` và host NVML)

---

## 📑 1. BẢN THIẾT KẾ ĐIỀU CHỈNH CHUẨN TẮC: STAIR5-v4 / NLGCL-CSE

| Trụ Cột Thành Phần | Cơ Chế Toán Học & Kỹ Thuật Thực Thi | Lợi Ích Vận Hành & Khắc Phục Lỗ Hổng Kỹ Thuật |
|:---|:---|:---|
| **1. Bảo Tồn 100% Backbone STAIR** | MI SVD Whitening, Tích chập phổ FSC $\beta = 1 - \beta_3$, toán tử BSC Smoother trong AdamWSEvo, hàm mất mát BPR, suy luận tích vô hướng Euclidean 100%. | Giữ nguyên vẹn độ trễ suy luận $O(1)$ (Zero Inference Overhead); bảo đảm baseline recovery chính xác khi $\eta=0$ hoặc arm $\in \{B0, N0\}$. |
| **2. Candidate Support Expansion (CSE)** | Khai thác tập cạnh đồng tương tác $c_{ij} = \sum_u R_{ui} R_{uj}$ với $c_{ij} \ge c_{\min}=2$ qua cơ chế block-wise sparse slicing. | Mở rộng tập cạnh ứng viên từ hành vi hợp tác huấn luyện (Collaborative Filtering support), khắc phục triệt để hạn chế kNN thuần túy của v3. |
| **3. Ochiai × Evidence Shrinkage** | $q_{ij} = \frac{c_{ij}}{c_{ij} + t} \cdot \frac{c_{ij}}{\sqrt{n_i n_j}}$ với $t=5.0$, lấy top-$k_{\text{CF}}=5$ per item với tie-breaking theo ID xác định. | Giảm thiểu nhiễu ngẫu nhiên từ các item ít tương tác; bảo đảm ma trận đối xứng chính xác $W_{\text{CF}} = \max(W_{\text{dir}}, W_{\text{dir}}^\top)$. |
| **4. Chuẩn Hóa Bậc & Isolated Fallback** | $\bar{S}_{\text{CF}} = S_{\text{CF}} + \text{diag}(\mathbf{1}[d_i^{\text{CF}} == 0])$ với chuẩn hóa đối xứng $(D_{\text{CF}})^{-1/2} W_{\text{CF}} (D_{\text{CF}})^{-1/2}$. | Item cô lập bảo toàn nguyên vẹn biểu diễn ego với trọng số 1.0; spectral norm luôn thỏa mãn $\|\bar{S}_{\text{CF}}\|_2 \le 1.0$. |
| **5. Toán Tử Lồi Tự Nhiên (Convex Blend)** | $S_4 = (1 - \eta) S_0 + \eta \bar{S}_{\text{CF}}$ (mặc định $\eta = 0.1$). Chặn phổ nghiêm ngặt $\|S_4\|_2 \le 1.0$. | Loại bỏ hoàn toàn phép chiếu KL nhân tạo gượng ép; $S_4$ đối xứng tự nhiên, bảo toàn năng lượng Laplacian. |
| **6. Chuẩn Tắc Hóa Đối Tương Phản NLGCL** | Cross-entity InfoNCE giữa user và item qua các tầng FSC trung gian ($G=1, \tau=0.2, \alpha=0.5, \lambda_{\text{nlgcl}}=0.01$). | Không tốn chi phí sinh view; tối ưu hóa không gian embedding đa tầng với anchor chunking chống tràn VRAM. |
| **7. Sửa Lỗi Cấu Hình Khoa Học** | Electronics dùng $\gamma=0.4$ (thay vì $0.2$), $\lambda_{\text{nlgcl}}=0.01$ (sửa lỗi default 0.1), batch_size=4096. | Đạt độ ổn định hội tụ tối ưu trên tập quy mô lớn; tuân thủ nghiệm thức thiết kế được công bố. |

```
                ┌────────────────────────────────────────────────────────┐
                │             STAIR5-v4 / NLGCL-CSE ARCHITECTURE         │
                └────────────────────────────────────────────────────────┘
                                             │
      ┌──────────────────────────────────────┴──────────────────────────────────────┐
      ▼                                                                             ▼
[CALIBRATED CSE OPERATOR]                                                 [RECOMMENDATION BACKBONE]
Modality Features ➔ Baseline kNN ➔ S0                                      User-Item Graph ➔ FSC Convolution
Train Interactions ➔ Block-wise Co-occurrence ➔ Shrinkage q_ij             Euclidean Dot-Product Scorer ➔ BPR
      ↓                                                                             ↓
Top-k CF Graph W_CF ➔ Normalized S_CF with Isolated Fallback               L_BPR Loss
      ↓                                                                             ↓
S4 = (1 - η) S0 + η S_CF                                                  Item Gradients Smoothed by S4
(Sparse CSR, bounded ||S4||_2 <= 1.0)                                               ↓
                                                                          [Faithful NLGCL InfoNCE Loss]
                                                                          L_total = L_BPR + λ_nlgcl * L_nlgcl
```


## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies & Đồng bộ Mã Nguồn STAIR5-v4
- Đồng bộ mã nguồn mới nhất từ GitHub repository `ThanhChuong12/STAIR-Enhanced` (branch `main`).
- Cài đặt các gói phụ thuộc chuẩn tắc tương thích với môi trường nghiệm thu: `freerec>=0.9.7`, `torchdata==0.8.0`, `torch-geometric`, `nvidia-ml-py`, `prettytable`, `matplotlib`, `pyyaml`, `scipy`, `pandas`.
- Khởi chạy tiến trình con xác thực (`Subprocess Environment Verification`) để kiểm tra toàn bộ imports và GPU runtime.


In [ ]:
# Cell 1: Môi trường, Dependencies & Đồng bộ STAIR-Enhanced (STAIR5-v4 NLGCL-CSE)
import os, shutil, subprocess, sys, types, glob, torch
from pathlib import Path

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working') if os.path.exists('/kaggle/working') else None

# 1. Đồng bộ repository STAIR-Enhanced thông minh & an toàn
if os.path.exists(STAIR_DIR):
    print("Thư mục STAIR-Enhanced đã tồn tại. Đang kiểm tra đồng bộ mã nguồn...")
    try:
        subprocess.run(['git', '-C', STAIR_DIR, 'fetch', 'origin', 'main'], check=True, capture_output=True)
        chk = subprocess.run(['git', '-C', STAIR_DIR, 'ls-tree', 'origin/main', 'models/stair5_v4.py'], capture_output=True, text=True)
        if 'models/stair5_v4.py' in chk.stdout:
            subprocess.run(['git', '-C', STAIR_DIR, 'reset', '--hard', 'origin/main'], check=True)
            print("✅ Đã đồng bộ về commit mới nhất của origin/main (chứa STAIR5-v4 NLGCL-CSE).")
        else:
            print("⚠️ Remote origin/main chưa có models/stair5_v4.py. Giữ nguyên mã nguồn hiện có trên đĩa.")
    except Exception as e:
        print(f"⚠️ Cảnh báo git fetch/check ({e}). Tiếp tục dùng mã nguồn hiện có trên đĩa.")

if not os.path.exists(STAIR_DIR) and os.path.exists('/kaggle/working'):
    print("Cloning STAIR-Enhanced repository (branch main)...")
    try:
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/ThanhChuong12/STAIR-Enhanced.git', STAIR_DIR
        ], check=True)
    except Exception as e:
        print(f"⚠️ Git clone gặp cảnh báo: {e}")

# Fallback: Tự động dò quét và sao chép mã nguồn từ Kaggle Input nếu chưa có models/stair5_v4.py
if not os.path.exists(os.path.join(STAIR_DIR, 'models', 'stair5_v4.py')):
    input_matches = glob.glob('/kaggle/input/**/models/stair5_v4.py', recursive=True)
    if input_matches:
        src_repo = os.path.dirname(os.path.dirname(input_matches[0]))
        print(f"📦 Phát hiện mã nguồn STAIR5-v4 từ Kaggle Input: {src_repo} -> Sao chép vào {STAIR_DIR}...")
        os.makedirs(STAIR_DIR, exist_ok=True)
        shutil.copytree(src_repo, STAIR_DIR, dirs_exist_ok=True)

active_dir = STAIR_DIR if os.path.exists(STAIR_DIR) else os.path.abspath('.')
for p in [active_dir, STAIR_DIR, '/kaggle/working', '.']:
    if p and os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

if os.path.exists(STAIR_DIR):
    os.chdir(STAIR_DIR)

# Thiết lập biến môi trường chuẩn deterministic và PYTHONPATH cho subprocess
sub_env = os.environ.copy()
sub_env['PYTHONPATH'] = f"{active_dir}:{os.environ.get('PYTHONPATH', '')}"
sub_env['PYTHONWARNINGS'] = 'ignore'
sub_env.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['PYTHONPATH'] = sub_env['PYTHONPATH']
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# Xóa cache module để kernel luôn nạp phiên bản v4 mới nhất từ đĩa
for mod_name in list(sys.modules.keys()):
    if any(k in mod_name for k in ['stair5_v4', 'models.stair5_v4', 'optimizers.stair5_v4_smoother']):
        sys.modules.pop(mod_name, None)

# 2. Cài đặt các gói phụ thuộc chuẩn tắc tương thích với môi trường nghiệm thu
print("Cài đặt dependencies tương thích (freerec, torchdata, torch-geometric, pyyaml, prettytable)...")
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'pynvml', 'torchdata'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.8.0'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'freerec>=0.9.7', 'torch-geometric', 'nvidia-ml-py', 'prettytable', 'matplotlib', 'pyyaml', 'scipy', 'pandas'
], check=False)

# 3. Patch tương thích nội bộ PyTorch Dynamo & TorchData DataPipe Registration
try:
    import models.freerec_compat
except Exception:
    try:
        import freerec_compat
    except Exception:
        pass

# 4. Xác minh môi trường trong tiến trình con (Subprocess Verification)
verify_script = '''
import torch, sys
print("=== SUBPROCESS ENVIRONMENT VERIFICATION ===")
print("Python Executable :", sys.executable)
print("Python Version    :", sys.version.split()[0])
print("PyTorch Version   :", torch.__version__)
print("CUDA Available    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name       :", torch.cuda.get_device_name(0))
    print("Device Count      :", torch.cuda.device_count())
    print("Allocated VRAM    : {:.2f} MB".format(torch.cuda.memory_allocated() / (1024**2)))
try:
    import freerec
    print("FreeRec Version   :", freerec.__version__)
except Exception as e:
    print("FreeRec Import    : FAILED ({})".format(e))
try:
    from models.stair5_v4 import STAIR5_v4_Model, ARMS_V4
    print("STAIR5-v4 Arms    :", list(ARMS_V4.keys()))
    print("STAIR5-v4 Core    : VERIFIED OK")
except Exception as e:
    print("STAIR5-v4 Core    : FAILED ({})".format(e))
print("===========================================")
'''
res = subprocess.run([sys.executable, '-c', verify_script], capture_output=True, text=True, env=sub_env)
print(res.stdout)
if res.stderr and "warning" not in res.stderr.lower():
    print("Stderr:", res.stderr)
print("✅ Khởi tạo môi trường STAIR5-v4 hoàn tất thành công.")


## Cell 2 📂 Chuẩn bị Dữ liệu từ Kaggle Input (Multi-Bridge sang /kaggle/data & Processed)
- Tự động dò quét toàn bộ kho dữ liệu Kaggle Input (`/kaggle/input`) để tìm kiếm 3 tập chuẩn: **Amazon Sports**, **Amazon Baby**, **Amazon Electronics**.
- Thiết lập cơ chế Symlink/Hardlink liên kết đa hướng sang `/kaggle/data`, `/kaggle/data/Processed`, `STAIR-Enhanced/data` để FreeRec luôn tìm thấy dữ liệu ở mọi vị trí.


In [ ]:
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input sang /kaggle/data & Processed
import os, shutil, glob

DATA_ROOT = '/kaggle/data'
PROCESSED_ROOT = os.path.join(DATA_ROOT, 'Processed')
LOCAL_DATA = '/kaggle/working/STAIR-Enhanced/data'
LOCAL_PROCESSED = os.path.join(LOCAL_DATA, 'Processed')

for d in [DATA_ROOT, PROCESSED_ROOT, LOCAL_DATA, LOCAL_PROCESSED]:
    os.makedirs(d, exist_ok=True)

TARGET_DATASETS = {
    'sports':      ('Amazon2014Sports_550_MMRec', ['sport', 'sports', 'amazon2014sports']),
    'baby':        ('Amazon2014Baby_550_MMRec', ['baby', 'amazon2014baby']),
    'electronics': ('Amazon2014Electronics_550_MMRec', ['electronic', 'electronics', 'amazon2014electronics']),
}

REQUIRED_EXTENSIONS = ('.npy', '.pkl', '.txt', '.inter', '.item', '.pt', '.csv', '.yaml')

def bridge_directories(src_dir, target_folder):
    '''Đồng bộ dữ liệu sang toàn bộ các vị trí FreeRec có thể tìm kiếm'''
    destinations = [
        os.path.join(DATA_ROOT, target_folder),
        os.path.join(PROCESSED_ROOT, target_folder),
        os.path.join(LOCAL_DATA, target_folder),
        os.path.join(LOCAL_PROCESSED, target_folder),
    ]
    for dst in destinations:
        if os.path.abspath(src_dir) == os.path.abspath(dst):
            continue
        os.makedirs(dst, exist_ok=True)
        for item in os.listdir(src_dir):
            s_item = os.path.join(src_dir, item)
            d_item = os.path.join(dst, item)
            if os.path.isdir(s_item):
                if not os.path.exists(d_item):
                    try:
                        os.symlink(s_item, d_item)
                    except Exception:
                        shutil.copytree(s_item, d_item, dirs_exist_ok=True)
            else:
                if not os.path.exists(d_item) or os.path.getsize(d_item) == 0:
                    try:
                        os.symlink(s_item, d_item)
                    except Exception:
                        shutil.copy2(s_item, d_item)

print("=" * 80)
print("TIẾN TRÌNH DÒ QUÉT & LIÊN KẾT DỮ LIỆU TỰ ĐỘNG CHO STAIR5-v4 NLGCL-CSE:")
prepared_data = set()
search_bases = ['/kaggle/input', '.', '..', '/kaggle/working']

for key, (folder_name, aliases) in TARGET_DATASETS.items():
    found_src = None
    for base in search_bases:
        if not os.path.exists(base):
            continue
        for root, dirs, files in os.walk(base):
            bname = os.path.basename(root).lower()
            if bname == folder_name.lower() or any(alias in bname for alias in aliases):
                has_req = any(f.endswith(REQUIRED_EXTENSIONS) for f in files)
                if has_req:
                    found_src = root
                    break
        if found_src:
            break

    if found_src:
        bridge_directories(found_src, folder_name)
        prepared_data.add(key)
        item_count = len(os.listdir(found_src))
        print(f"  ✅ [SẴN SÀNG] {key.upper():12s} -> Nguồn: {found_src} ({item_count} files)")
    else:
        print(f"  ⚠️ [CHƯA THẤY] {key.upper():12s} -> Sẽ nạp tự động qua kịch bản hoặc config.")
print("=" * 80)


## Cell 3 🧪 Bộ Kiểm Thử Độc Lập Toàn Diện Cho STAIR5-v4 NLGCL-CSE (11 Tests)
- Thực thi nghiệm thu toán học và pipeline độc lập trước khi huấn luyện.
- Kiểm tra tính toán shrinking score Ochiai, cấu trúc ma trận CF thưa theo khối, fallback item cô lập, chặn phổ toán tử lồi $\|S_4\|_2 \le 1.0$, tính tương đương số học InfoNCE chunked, và chu trình huấn luyện khép kín (forward/backward/optimizer step).


In [ ]:
# Cell 3: Chạy Bộ Kiểm Thử Độc Lập Toàn Diện Cho STAIR5-v4 NLGCL-CSE
import os, sys, subprocess

print("=" * 80)
print("🚀 CHẠY BỘ KIỂM THỬ ĐỘC LẬP TOÀN DIỆN CHO STAIR5-v4 NLGCL-CSE (11 TESTS)...")
print("=" * 80)

test_files = [
    "tests/test_stair5_v4_graph.py",
    "tests/test_stair5_v4_objectives.py",
    "tests/test_stair5_v4_pipeline.py",
]

total_passed = 0
total_failed = 0

sub_env = os.environ.copy()
sub_env['PYTHONPATH'] = f".:{os.environ.get('PYTHONPATH', '')}"

for tf in test_files:
    if os.path.exists(tf):
        print(f"--> Đang chạy test suite: {tf} ...")
        cmd = [sys.executable, tf]
        res = subprocess.run(cmd, capture_output=True, text=True, env=sub_env)
        out = (res.stdout + "\n" + res.stderr).strip()
        print(out)
        if res.returncode == 0:
            print(f"✅ {tf}: PASS")
            total_passed += 1
        else:
            print(f"❌ {tf}: FAILED (code {res.returncode})")
            total_failed += 1
    else:
        print(f"⚠️ Không tìm thấy file: {tf}")
        total_failed += 1

print("=" * 80)
if total_failed == 0:
    print(f"🎉 TẤT CẢ CÁC BỘ TEST CỦA STAIR5-v4 ĐÃ VƯỢT QUA XUẤT SẮC ({total_passed}/{len(test_files)} suites)! SẴN SÀNG HUẤN LUYỆN.")
else:
    print(f"⚠️ Cảnh báo: Có {total_failed} suite kiểm thử không thành công. Hãy kiểm tra logs chi tiết ở trên.")
print("=" * 80)


## Cell 4 🛠️ Telemetry Engine: Training Runner, GPU VRAM Profiler & Visualizer
- Quản lý tiến trình huấn luyện `main_stair5_v4.py` trong sub-process an toàn, cô lập hoàn toàn bộ nhớ.
- Giám sát VRAM phần cứng thời gian thực qua background thread (`nvidia-ml-py` / NVML).
- Trích xuất tự động `training_telemetry.jsonl` và checkpoint tốt nhất (`[Coach] >>> Load best model`).


In [ ]:
# Cell 4: Telemetry Engine — Training Runner, Hardware Profiler & Visualization
import os, sys, time, re, json, threading, subprocess
from pathlib import Path
import numpy as np

sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None

TRACKED_METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

BASELINE_REF = {
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V1_H0_REF = {
    'sports':      {'Recall@10': 0.0747, 'Recall@20': 0.1133, 'NDCG@10': 0.0407, 'NDCG@20': 0.0506},
    'baby':        {'Recall@10': 0.0660, 'Recall@20': 0.1030, 'NDCG@10': 0.0352, 'NDCG@20': 0.0447},
    'electronics': {'Recall@10': 0.0435, 'Recall@20': 0.0666, 'NDCG@10': 0.0241, 'NDCG@20': 0.0301},
}

V2_REF = {
    'sports':      {'Recall@10': 0.0753, 'Recall@20': 0.1113, 'NDCG@10': 0.0415, 'NDCG@20': 0.0508},
    'baby':        {'Recall@10': 0.0669, 'Recall@20': 0.1027, 'NDCG@10': 0.0362, 'NDCG@20': 0.0454},
    'electronics': {'Recall@10': 0.0451, 'Recall@20': 0.0678, 'NDCG@10': 0.0252, 'NDCG@20': 0.0311},
}

V3_REF = {
    'sports':      {'Recall@10': 0.0758, 'Recall@20': 0.1122, 'NDCG@10': 0.0418, 'NDCG@20': 0.0512},
    'baby':        {'Recall@10': 0.0678, 'Recall@20': 0.1030, 'NDCG@10': 0.0362, 'NDCG@20': 0.0452},
    'electronics': {'Recall@10': 0.0460, 'Recall@20': 0.0685, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
}

DATASET_PROFILES = {
    'sports': {
        'name':        'Amazon Sports',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '35,598 Users | 18,357 Items | 296K Interactions',
        'color':       '#ff7f0e',
        'approx_mins': 45.0,
    },
    'baby': {
        'name':        'Amazon Baby',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '19,445 Users | 7,050 Items | 160K Interactions',
        'color':       '#1f77b4',
        'approx_mins': 20.0,
    },
    'electronics': {
        'name':        'Amazon Electronics',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '192,403 Users | 63,001 Items | 1.69M Interactions',
        'color':       '#2ca02c',
        'approx_mins': 260.0,
    },
}

vram_profile = {}

def vram_monitor(key, stop_evt, interval=2.0):
    '''Background thread tracking host GPU allocated memory via NVML.'''
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        records = []
        while not stop_evt.is_set():
            mem = pynvml.nvmlDeviceGetMemoryInfo(h)
            records.append(mem.used / (1024**2))
            time.sleep(interval)
        pynvml.nvmlShutdown()
        vram_profile[key] = records
    except Exception:
        vram_profile[key] = []

def resolve_path(p):
    if os.path.exists(p):
        return p
    base = os.path.basename(p)
    candidates = [
        p,
        os.path.join('/kaggle/working/logs/stair5_v4', base),
        os.path.join('/kaggle/working/logs/GD5', base),
        os.path.join('logs/stair5_v4', base),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return p

def parse_telemetry_jsonl(artifact_dir):
    p = Path(artifact_dir) / "training_telemetry.jsonl"
    if not p.is_file():
        return []
    records = []
    with p.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    return records

def extract_best_validation(log_path):
    '''Extracts best validation epoch and validation metrics.'''
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    best_val_epoch = None
    best_val_metrics = {}
    best_ndcg20 = -1.0

    matches = re.finditer(r'VALID\s+@Epoch:\s*(\d+)(.*?)(?=(?:VALID|TEST|Load|TRAIN|\Z))', content, re.DOTALL)
    for m in matches:
        ep = int(m.group(1))
        block = m.group(2)
        ndcg_m = re.search(r'NDCG@20(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', block, re.IGNORECASE)
        if ndcg_m:
            val_ndcg20 = float(ndcg_m.group(1))
            if val_ndcg20 > best_ndcg20:
                best_ndcg20 = val_ndcg20
                best_val_epoch = ep
                cur_metrics = {'NDCG@20': val_ndcg20}
                for metric in ['Recall@10', 'Recall@20', 'NDCG@10']:
                    mm = re.search(rf'{metric}(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', block, re.IGNORECASE)
                    if mm:
                        cur_metrics[metric] = float(mm.group(1))
                best_val_metrics = cur_metrics
    return best_val_epoch, best_val_metrics

def extract_test_metrics(log_path):
    '''Strictly extracts test evaluation metrics ONLY after checkpoint load.'''
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    lines = content.splitlines()
    tail = '\n'.join(lines[-35:])
    if 'Traceback (most recent call last):' in tail:
        return None, {}

    best_match = re.search(r'\[Coach\]\s*>>>\s*Load best model\s*@Epoch:\s*(\d+)', content)
    if not best_match:
        return None, {}

    best_model_epoch = int(best_match.group(1))
    content_after_load = content[best_match.end():]

    test_match = re.search(r'TEST\s+@Epoch:\s*(\d+)(.*?)(?:=================|$)', content_after_load, re.DOTALL)
    if not test_match:
        return None, {}

    test_epoch = int(test_match.group(1))
    if test_epoch != best_model_epoch:
        return None, {}

    test_snippet = test_match.group(2)
    test_metrics = {}
    for metric in TRACKED_METRICS:
        m = re.search(rf'{metric}(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', test_snippet, re.IGNORECASE)
        if m:
            test_metrics[metric] = float(m.group(1))

    if len(test_metrics) < len(TRACKED_METRICS):
        return None, {}
    return test_epoch, test_metrics

def parse_training_losses(log_path, telemetry=None):
    '''Extracts pure BPR loss and total training loss trajectories.'''
    if telemetry and len(telemetry) > 0:
        bpr_losses = [(r['epoch'], float(r['bpr_loss'])) for r in telemetry if 'bpr_loss' in r]
        total_losses = [(r['epoch'], float(r['bpr_loss']) + float(r.get('lambda_nlgcl', 0.01)) * float(r.get('nlgcl_loss', 0.0))) for r in telemetry if 'bpr_loss' in r]
        return bpr_losses, total_losses

    log_path = resolve_path(log_path)
    bpr_losses, total_losses = [], []
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        telemetry_matches = re.findall(
            r'\[Epoch\s+(\d+)\s+Telemetry\]\s+BPR:\s*([0-9.]+)\s+\|\s+NLGCL:\s*([0-9.]+)',
            content
        )
        if telemetry_matches:
            for ep_str, bpr_str, nlgcl_str in telemetry_matches:
                bpr_losses.append((int(ep_str), float(bpr_str)))
        loss_matches = re.findall(r'TRAIN @Epoch:\s*(\d+).*?LOSS\s+Avg:\s*([0-9.]+)', content, re.DOTALL)
        total_losses = [(int(ep), float(loss)) for ep, loss in loss_matches]
    return bpr_losses, total_losses

def parse_valid_metric(log_path, metric='NDCG@20'):
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    pattern = rf'VALID\s+@Epoch:\s*(\d+).*?{metric}\s+Avg:\s*([0-9.]+)'
    matches = re.findall(pattern, content, re.IGNORECASE)
    return [(int(ep), float(v)) for ep, v in matches]

def run_training_stair5_v4(key, config_yaml=None, data_root='/kaggle/data', log_path=None, artifact_dir=None, **kwargs):
    '''Executes STAIR5-v4 NLGCL-CSE training with CLI argument overrides and strict error handling.'''
    cfg_dict = STAIR5_V4_CONFIGS.get(key, {}).copy()
    cfg_dict.update(kwargs)

    config_yaml = config_yaml or cfg_dict.get('yaml')
    log_path = log_path or cfg_dict.get('log')
    artifact_dir = artifact_dir or cfg_dict.get('artifact_dir')

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    os.makedirs(artifact_dir, exist_ok=True)

    print('=' * 80)
    print(f'🚀 KHỞI ĐỘNG TIẾN TRÌNH HUẤN LUYỆN STAIR5-v4 / NLGCL-CSE: {key.upper()}')
    print(f'  * Dataset Key         : {key}')
    print(f'  * YAML Configuration  : {config_yaml}')
    print(f'  * Log Path            : {log_path}')
    print(f'  * Artifact Directory  : {artifact_dir}')
    print(f'  * V4 Arm              : {cfg_dict.get("v4_arm", "N-CSE")}')
    print(f'  * Convex Blend (η)    : {cfg_dict.get("eta", 0.1)}')
    print(f'  * Top-k CF (k_cf)     : {cfg_dict.get("k_cf", 5)}')
    print(f'  * Min Co-occur (c_min): {cfg_dict.get("c_min", 2)}')
    print(f'  * Shrinkage (t)       : {cfg_dict.get("t_shrinkage", 5.0)}')
    print(f'  * NLGCL Weight (λ)    : {cfg_dict.get("lambda_nlgcl", 0.01)}')
    print(f'  * NLGCL Temp (τ)      : {cfg_dict.get("tau_nlgcl", 0.2)}')
    print(f'  * Seed                : {cfg_dict.get("seed", 1)}')
    print('=' * 80)

    stop_evt = threading.Event()
    th = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    th.start()

    t0 = time.time()
    runner_py = '/kaggle/working/STAIR-Enhanced/main_stair5_v4.py'
    if not os.path.exists(runner_py):
        runner_py = 'main_stair5_v4.py'

    if not os.path.exists(config_yaml):
        cand_y = os.path.join('configs', os.path.basename(config_yaml))
        if os.path.exists(cand_y):
            config_yaml = cand_y

    device_str = str(cfg_dict.get('device', '0' if torch.cuda.is_available() else 'cpu'))
    cmd = [
        sys.executable, '-u', runner_py,
        '--config', str(config_yaml),
        '--root', str(data_root),
        '--v4-arm', str(cfg_dict.get('v4_arm', 'N-CSE')),
        '--seed', str(cfg_dict.get('seed', 1)),
        '--device', device_str,
        '--ranking', 'full',
        '--eval-freq', str(cfg_dict.get('eval_freq', 5)),
        '--eta', str(cfg_dict.get('eta', 0.1)),
        '--k-cf', str(cfg_dict.get('k_cf', 5)),
        '--c-min', str(cfg_dict.get('c_min', 2)),
        '--t-shrinkage', str(cfg_dict.get('t_shrinkage', 5.0)),
        '--cf-block-size', str(cfg_dict.get('cf_block_size', 1024)),
        '--lambda-nlgcl', str(cfg_dict.get('lambda_nlgcl', 0.01)),
        '--tau-nlgcl', str(cfg_dict.get('tau_nlgcl', 0.2)),
        '--g-nlgcl', str(cfg_dict.get('g_nlgcl', 1)),
        '--alpha-nlgcl', str(cfg_dict.get('alpha_nlgcl', 0.5)),
        '--nlgcl-chunk-size', str(cfg_dict.get('nlgcl_chunk_size', 1024)),
        '--artifact-dir', str(artifact_dir),
        '--log', str(log_path),
    ]

    env = os.environ.copy()
    env['PYTHONPATH'] = f".:{os.environ.get('PYTHONPATH', '')}"
    env['PYTHONWARNINGS'] = 'ignore'

    try:
        with open(log_path, 'w', encoding='utf-8') as lf:
            proc = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                env=env,
            )
            for line in proc.stdout:
                sys.stdout.write(line)
                sys.stdout.flush()
                lf.write(line)
                lf.flush()
            proc.wait()
            ret_code = proc.returncode
    finally:
        stop_evt.set()
        th.join(timeout=3.0)

    elapsed = time.time() - t0
    print(f'⏱️ Thời gian thực thi ({key.upper()}): {elapsed/60:.2f} phút | Trạng thái hoàn thành: {ret_code}')
    if ret_code != 0:
        raise RuntimeError(f'Tiến trình huấn luyện {key.upper()} thất bại với mã lỗi {ret_code}. Kiểm tra log tại: {log_path}')
    return elapsed


## Cell 5 📋 Cấu hình Siêu tham số STAIR5-v4 NLGCL-CSE (Dataset-Adaptive Matrix)
- Ma trận cấu hình chi tiết cho 3 tập dữ liệu theo đặc tả toán học trong `docs/giai_doan_5/STAIR5_v4_Report.md`:
  * **Amazon Sports**: $\gamma = 0.2, wd = 0.1, \text{lr} = 0.001, \eta = 0.1, k_{\text{CF}} = 5, \lambda_{\text{nlgcl}} = 0.01$.
  * **Amazon Baby**: $\gamma = 0.1, wd = 0.3, \text{lr} = 0.001, \eta = 0.1, k_{\text{CF}} = 5, \lambda_{\text{nlgcl}} = 0.01$.
  * **Amazon Electronics**: $\gamma = 0.4$ *(hiệu chỉnh)*, $wd = 0.1, \text{lr} = 0.001, \text{batch\_size} = 4096, \eta = 0.1, k_{\text{CF}} = 5, \lambda_{\text{nlgcl}} = 0.01$.


In [ ]:
# Cell 5: Cấu hình Siêu tham số STAIR5-v4 NLGCL-CSE (Dataset-Adaptive Matrix)
import os

os.makedirs('/kaggle/working/logs/stair5_v4', exist_ok=True)
os.makedirs('/kaggle/working/reports', exist_ok=True)
os.makedirs('/kaggle/working/runs/stair5_v4/graph_cache', exist_ok=True)

STAIR5_V4_CONFIGS = {
    'sports': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Sports_STAIR5_v4.yaml',
        'log':                '/kaggle/working/logs/stair5_v4/sports_stair5_v4.log',
        'artifact_dir':       '/kaggle/working/runs/stair5_v4/sports_N-CSE_seed1',
        'v4_arm':             'N-CSE',
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      1024,
        'lambda_nlgcl':       0.01,
        'tau_nlgcl':          0.2,
        'g_nlgcl':            1,
        'alpha_nlgcl':        0.5,
        'nlgcl_chunk_size':   1024,
        'seed':               1,
        'eval_freq':          5,
    },
    'baby': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Baby_STAIR5_v4.yaml',
        'log':                '/kaggle/working/logs/stair5_v4/baby_stair5_v4.log',
        'artifact_dir':       '/kaggle/working/runs/stair5_v4/baby_N-CSE_seed1',
        'v4_arm':             'N-CSE',
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      1024,
        'lambda_nlgcl':       0.01,
        'tau_nlgcl':          0.2,
        'g_nlgcl':            1,
        'alpha_nlgcl':        0.5,
        'nlgcl_chunk_size':   1024,
        'seed':               1,
        'eval_freq':          5,
    },
    'electronics': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Electronics_STAIR5_v4.yaml',
        'log':                '/kaggle/working/logs/stair5_v4/electronics_stair5_v4.log',
        'artifact_dir':       '/kaggle/working/runs/stair5_v4/electronics_N-CSE_seed1',
        'v4_arm':             'N-CSE',
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      2048,
        'lambda_nlgcl':       0.01,
        'tau_nlgcl':          0.2,
        'g_nlgcl':            1,
        'alpha_nlgcl':        0.5,
        'nlgcl_chunk_size':   1024,
        'seed':               1,
        'eval_freq':          5,
    },
}

print('✅ Ma trận cấu hình STAIR5-v4 NLGCL-CSE đã nạp thành công:')
for k, v in STAIR5_V4_CONFIGS.items():
    print(f"  * {k.upper():12s} -> Arm: {v['v4_arm']} | Config: {os.path.basename(v['yaml'])} | Log: {os.path.basename(v['log'])}")


## Cell 6a 🏋️ Huấn luyện Pha A — Amazon Sports (Pilot & Đề Xuất Chính N-CSE)
- Chạy huấn luyện mô hình STAIR5-v4 trên tập **Amazon Sports** (35,598 Users, 18,357 Items).
- Ghi nhận telemetry từng epoch: BPR loss, NLGCL InfoNCE loss, VRAM phân bổ, và thời gian thực thi.


In [ ]:
# Cell 6a: Training STAIR5-v4 NLGCL-CSE on Amazon Sports (Pha A: Pilot)
sports_time = run_training_stair5_v4('sports')
ep_sports, metrics_sports = extract_test_metrics(STAIR5_V4_CONFIGS['sports']['log'])
val_ep_sports, val_m_sports = extract_best_validation(STAIR5_V4_CONFIGS['sports']['log'])

print('\n' + '=' * 80)
print('📊 KẾT QUẢ THỰC NGHIỆM PHA A — AMAZON SPORTS (STAIR5-v4 / N-CSE):')
print(f'  * Checkpoint tối ưu tại Epoch: {ep_sports} (Validation NDCG@20 = {val_m_sports.get("NDCG@20", 0.0):.4f})')
for m in TRACKED_METRICS:
    val = metrics_sports.get(m, 0.0)
    base = BASELINE_REF['sports'].get(m, 0.0)
    delta = ((val - base) / base) * 100 if base > 0 else 0.0
    print(f'  * {m:10s} : {val:.4f}  (Baseline STAIR: {base:.4f} | Biến thiên: {delta:+.2f}%)')
print('=' * 80)


## Cell 6b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Sports
- Phân tích chi tiết mức tiêu thụ VRAM thực tế từ `training_telemetry.jsonl` và NVML monitor.


In [ ]:
# Cell 6b: Model Tensor VRAM Profile — Amazon Sports (Paper Standard)
import matplotlib.pyplot as plt

sports_art = STAIR5_V4_CONFIGS['sports']['artifact_dir']
telemetry_sports = parse_telemetry_jsonl(sports_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if telemetry_sports:
    eps = [r['epoch'] for r in telemetry_sports]
    vram_mb = [r.get('vram_allocated_mb', 0.0) for r in telemetry_sports]
    plt.plot(eps, vram_mb, color='#ff7f0e', lw=2.2, marker='o', ms=4, label='Peak Allocated VRAM (MB)')
    plt.title('Amazon Sports — Peak Model Tensor VRAM Profile (STAIR5-v4)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'sports' in vram_profile and len(vram_profile['sports']) > 0:
    plt.plot(vram_profile['sports'], color='#ff7f0e', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Sports — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    plt.text(0.5, 0.5, 'Chưa có dữ liệu VRAM hoặc NVML không khả dụng', ha='center', va='center')

plt.tight_layout()
os.makedirs('/kaggle/working/reports', exist_ok=True)
plt.savefig('/kaggle/working/reports/vram_profile_sports.png')
plt.show()


## Cell 6c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Sports
- Trực quan hóa đường cong hàm mất mát BPR và đường cong đánh giá Validation NDCG@20 theo từng epoch.


In [ ]:
# Cell 6c: Learning Dynamics & Convergence Profiles — Amazon Sports
fig, ax1 = plt.subplots(figsize=(9, 4.5), dpi=150)

log_sports = STAIR5_V4_CONFIGS['sports']['log']
bpr_losses, total_losses = parse_training_losses(log_sports, telemetry_sports)
val_ndcg = parse_valid_metric(log_sports, 'NDCG@20')

if total_losses:
    x_loss, y_loss = zip(*total_losses)
    ax1.plot(x_loss, y_loss, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if bpr_losses:
    x_bpr, y_bpr = zip(*bpr_losses)
    ax1.plot(x_bpr, y_bpr, color='#1f77b4', lw=1.5, ls='--', alpha=0.7, label='Pure BPR Loss')

ax1.set_xlabel('Epoch', fontsize=10)
ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.grid(True, linestyle='--', alpha=0.4)

if val_ndcg:
    ax2 = ax1.twinx()
    x_val, y_val = zip(*val_ndcg)
    ax2.plot(x_val, y_val, color='#d62728', lw=2.2, marker='s', ms=4, label='Validation NDCG@20')
    ax2.set_ylabel('Validation NDCG@20', color='#d62728', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='#d62728')
    if 'sports' in BASELINE_REF:
        ax2.axhline(BASELINE_REF['sports']['NDCG@20'], color='gray', ls=':', label='Baseline STAIR')

plt.title('Amazon Sports — Learning Dynamics (STAIR5-v4 NLGCL-CSE)', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_sports.png')
plt.show()


## Cell 7a 🏋️ Huấn luyện Pha B — Amazon Baby (Kiểm chứng An toàn)
- Chạy huấn luyện mô hình STAIR5-v4 trên tập **Amazon Baby** (19,445 Users, 7,050 Items).
- Kiểm tra tính ổn định trên tập mật độ thưa cao với siêu tham số $\gamma=0.1, wd=0.3$.


In [ ]:
# Cell 7a: Training STAIR5-v4 NLGCL-CSE on Amazon Baby (Pha B: Confirmatory)
baby_time = run_training_stair5_v4('baby')
ep_baby, metrics_baby = extract_test_metrics(STAIR5_V4_CONFIGS['baby']['log'])
val_ep_baby, val_m_baby = extract_best_validation(STAIR5_V4_CONFIGS['baby']['log'])

print('\n' + '=' * 80)
print('📊 KẾT QUẢ THỰC NGHIỆM PHA B — AMAZON BABY (STAIR5-v4 / N-CSE):')
print(f'  * Checkpoint tối ưu tại Epoch: {ep_baby} (Validation NDCG@20 = {val_m_baby.get("NDCG@20", 0.0):.4f})')
for m in TRACKED_METRICS:
    val = metrics_baby.get(m, 0.0)
    base = BASELINE_REF['baby'].get(m, 0.0)
    delta = ((val - base) / base) * 100 if base > 0 else 0.0
    print(f'  * {m:10s} : {val:.4f}  (Baseline STAIR: {base:.4f} | Biến thiên: {delta:+.2f}%)')
print('=' * 80)


## Cell 7b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Baby
- Phân tích chi tiết mức tiêu thụ VRAM thực tế trên Amazon Baby.


In [ ]:
# Cell 7b: Model Tensor VRAM Profile — Amazon Baby (Paper Standard)
baby_art = STAIR5_V4_CONFIGS['baby']['artifact_dir']
telemetry_baby = parse_telemetry_jsonl(baby_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if telemetry_baby:
    eps = [r['epoch'] for r in telemetry_baby]
    vram_mb = [r.get('vram_allocated_mb', 0.0) for r in telemetry_baby]
    plt.plot(eps, vram_mb, color='#1f77b4', lw=2.2, marker='o', ms=4, label='Peak Allocated VRAM (MB)')
    plt.title('Amazon Baby — Peak Model Tensor VRAM Profile (STAIR5-v4)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'baby' in vram_profile and len(vram_profile['baby']) > 0:
    plt.plot(vram_profile['baby'], color='#1f77b4', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Baby — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    plt.text(0.5, 0.5, 'Chưa có dữ liệu VRAM hoặc NVML không khả dụng', ha='center', va='center')

plt.tight_layout()
plt.savefig('/kaggle/working/reports/vram_profile_baby.png')
plt.show()


## Cell 7c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Baby
- Trực quan hóa đường cong hàm mất mát BPR và đường cong đánh giá Validation NDCG@20 trên Amazon Baby.


In [ ]:
# Cell 7c: Learning Dynamics & Convergence Profiles — Amazon Baby
fig, ax1 = plt.subplots(figsize=(9, 4.5), dpi=150)

log_baby = STAIR5_V4_CONFIGS['baby']['log']
bpr_losses_b, total_losses_b = parse_training_losses(log_baby, telemetry_baby)
val_ndcg_b = parse_valid_metric(log_baby, 'NDCG@20')

if total_losses_b:
    x_loss, y_loss = zip(*total_losses_b)
    ax1.plot(x_loss, y_loss, color='#ff7f0e', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if bpr_losses_b:
    x_bpr, y_bpr = zip(*bpr_losses_b)
    ax1.plot(x_bpr, y_bpr, color='#ff7f0e', lw=1.5, ls='--', alpha=0.7, label='Pure BPR Loss')

ax1.set_xlabel('Epoch', fontsize=10)
ax1.set_ylabel('Training Loss', color='#ff7f0e', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#ff7f0e')
ax1.grid(True, linestyle='--', alpha=0.4)

if val_ndcg_b:
    ax2 = ax1.twinx()
    x_val, y_val = zip(*val_ndcg_b)
    ax2.plot(x_val, y_val, color='#2ca02c', lw=2.2, marker='s', ms=4, label='Validation NDCG@20')
    ax2.set_ylabel('Validation NDCG@20', color='#2ca02c', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='#2ca02c')
    if 'baby' in BASELINE_REF:
        ax2.axhline(BASELINE_REF['baby']['NDCG@20'], color='gray', ls=':', label='Baseline STAIR')

plt.title('Amazon Baby — Learning Dynamics (STAIR5-v4 NLGCL-CSE)', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_baby.png')
plt.show()


## Cell 8a 🏋️ Huấn luyện Pha C — Amazon Electronics (~1.7M Tương tác)
- Kiểm chứng khả năng mở rộng (Scalability) của STAIR5-v4 trên tập dữ liệu quy mô lớn **Amazon Electronics** (192,403 Users, 63,001 Items, 1.69M Interactions).
- Sử dụng cấu hình chuẩn tắc đã sửa đổi: $\gamma = 0.4, \text{batch\_size} = 4096, \text{block\_size} = 2048, \text{nlgcl\_chunk\_size} = 1024$.


In [ ]:
# Cell 8a: Training STAIR5-v4 NLGCL-CSE on Amazon Electronics (Pha C: Scale-up)
electronics_time = run_training_stair5_v4('electronics')
ep_elec, metrics_elec = extract_test_metrics(STAIR5_V4_CONFIGS['electronics']['log'])
val_ep_elec, val_m_elec = extract_best_validation(STAIR5_V4_CONFIGS['electronics']['log'])

print('\n' + '=' * 80)
print('📊 KẾT QUẢ THỰC NGHIỆM PHA C — AMAZON ELECTRONICS (STAIR5-v4 / N-CSE):')
print(f'  * Checkpoint tối ưu tại Epoch: {ep_elec} (Validation NDCG@20 = {val_m_elec.get("NDCG@20", 0.0):.4f})')
for m in TRACKED_METRICS:
    val = metrics_elec.get(m, 0.0)
    base = BASELINE_REF['electronics'].get(m, 0.0)
    delta = ((val - base) / base) * 100 if base > 0 else 0.0
    print(f'  * {m:10s} : {val:.4f}  (Baseline STAIR: {base:.4f} | Biến thiên: {delta:+.2f}%)')
print('=' * 80)


## Cell 8b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Electronics
- Phân tích chi tiết mức tiêu thụ VRAM thực tế trên Amazon Electronics.


In [ ]:
# Cell 8b: Model Tensor VRAM Profile — Amazon Electronics (Paper Standard)
elec_art = STAIR5_V4_CONFIGS['electronics']['artifact_dir']
telemetry_elec = parse_telemetry_jsonl(elec_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if telemetry_elec:
    eps = [r['epoch'] for r in telemetry_elec]
    vram_mb = [r.get('vram_allocated_mb', 0.0) for r in telemetry_elec]
    plt.plot(eps, vram_mb, color='#2ca02c', lw=2.2, marker='o', ms=4, label='Peak Allocated VRAM (MB)')
    plt.title('Amazon Electronics — Peak Model Tensor VRAM Profile (STAIR5-v4)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'electronics' in vram_profile and len(vram_profile['electronics']) > 0:
    plt.plot(vram_profile['electronics'], color='#2ca02c', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Electronics — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    plt.text(0.5, 0.5, 'Chưa có dữ liệu VRAM hoặc NVML không khả dụng', ha='center', va='center')

plt.tight_layout()
plt.savefig('/kaggle/working/reports/vram_profile_electronics.png')
plt.show()


## Cell 8c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Electronics
- Trực quan hóa đường cong hàm mất mát BPR và đường cong đánh giá Validation NDCG@20 trên Amazon Electronics.


In [ ]:
# Cell 8c: Learning Dynamics & Convergence Profiles — Amazon Electronics
fig, ax1 = plt.subplots(figsize=(9, 4.5), dpi=150)

log_elec = STAIR5_V4_CONFIGS['electronics']['log']
bpr_losses_e, total_losses_e = parse_training_losses(log_elec, telemetry_elec)
val_ndcg_e = parse_valid_metric(log_elec, 'NDCG@20')

if total_losses_e:
    x_loss, y_loss = zip(*total_losses_e)
    ax1.plot(x_loss, y_loss, color='#2ca02c', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if bpr_losses_e:
    x_bpr, y_bpr = zip(*bpr_losses_e)
    ax1.plot(x_bpr, y_bpr, color='#2ca02c', lw=1.5, ls='--', alpha=0.7, label='Pure BPR Loss')

ax1.set_xlabel('Epoch', fontsize=10)
ax1.set_ylabel('Training Loss', color='#2ca02c', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#2ca02c')
ax1.grid(True, linestyle='--', alpha=0.4)

if val_ndcg_e:
    ax2 = ax1.twinx()
    x_val, y_val = zip(*val_ndcg_e)
    ax2.plot(x_val, y_val, color='#9467bd', lw=2.2, marker='s', ms=4, label='Validation NDCG@20')
    ax2.set_ylabel('Validation NDCG@20', color='#9467bd', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='#9467bd')
    if 'electronics' in BASELINE_REF:
        ax2.axhline(BASELINE_REF['electronics']['NDCG@20'], color='gray', ls=':', label='Baseline STAIR')

plt.title('Amazon Electronics — Learning Dynamics (STAIR5-v4 NLGCL-CSE)', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_electronics.png')
plt.show()


## Cell 9 📊 Bảng So sánh Tổng hợp Ablation Study Đa Phiên bản (Baseline vs v1 vs v2 vs v3 vs v4)
- Lập bảng so sánh chi tiết tất cả các chỉ số đo lường: `Recall@10`, `Recall@20`, `NDCG@10`, `NDCG@20`, Peak VRAM, và % cải thiện tương đối.


In [ ]:
# Cell 9: Bảng so sánh Ablation Study toàn diện (Recall@10, Recall@20, NDCG@10, NDCG@20)
from prettytable import PrettyTable

table = PrettyTable()
table.field_names = [
    "Tập Dữ Liệu", "Phiên Bản", "Recall@10", "Recall@20", "NDCG@10", "NDCG@20", "Δ NDCG@20 (%)"
]

all_results = {
    'sports': metrics_sports if 'metrics_sports' in locals() and metrics_sports else {},
    'baby': metrics_baby if 'metrics_baby' in locals() and metrics_baby else {},
    'electronics': metrics_elec if 'metrics_elec' in locals() and metrics_elec else {},
}

for ds in ['sports', 'baby', 'electronics']:
    b = BASELINE_REF[ds]
    table.add_row([ds.upper(), "Baseline STAIR", f"{b['Recall@10']:.4f}", f"{b['Recall@20']:.4f}", f"{b['NDCG@10']:.4f}", f"{b['NDCG@20']:.4f}", "0.00%"])

    v1 = V1_H0_REF[ds]
    d_v1 = ((v1['NDCG@20'] - b['NDCG@20']) / b['NDCG@20']) * 100
    table.add_row([ds.upper(), "STAIR5-v1 (H0)", f"{v1['Recall@10']:.4f}", f"{v1['Recall@20']:.4f}", f"{v1['NDCG@10']:.4f}", f"{v1['NDCG@20']:.4f}", f"{d_v1:+.2f}%"])

    v2 = V2_REF[ds]
    d_v2 = ((v2['NDCG@20'] - b['NDCG@20']) / b['NDCG@20']) * 100
    table.add_row([ds.upper(), "STAIR5-v2 (C-HET)", f"{v2['Recall@10']:.4f}", f"{v2['Recall@20']:.4f}", f"{v2['NDCG@10']:.4f}", f"{v2['NDCG@20']:.4f}", f"{d_v2:+.2f}%"])

    v3 = V3_REF[ds]
    d_v3 = ((v3['NDCG@20'] - b['NDCG@20']) / b['NDCG@20']) * 100
    table.add_row([ds.upper(), "STAIR5-v3 (DP-PC)", f"{v3['Recall@10']:.4f}", f"{v3['Recall@20']:.4f}", f"{v3['NDCG@10']:.4f}", f"{v3['NDCG@20']:.4f}", f"{d_v3:+.2f}%"])

    v4_m = all_results.get(ds, {})
    if v4_m and 'NDCG@20' in v4_m:
        d_v4 = ((v4_m['NDCG@20'] - b['NDCG@20']) / b['NDCG@20']) * 100
        table.add_row([ds.upper(), "STAIR5-v4 (N-CSE)", f"{v4_m['Recall@10']:.4f}", f"{v4_m['Recall@20']:.4f}", f"{v4_m['NDCG@10']:.4f}", f"{v4_m['NDCG@20']:.4f}", f"{d_v4:+.2f}%"])
    else:
        table.add_row([ds.upper(), "STAIR5-v4 (N-CSE)", "Đang huấn luyện", "-", "-", "-", "-"])

print("=" * 95)
print("📊 BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG ĐA PHIÊN BẢN (BASELINE ➔ v1 ➔ v2 ➔ v3 ➔ v4):")
print(table)
print("=" * 95)


## Cell 10 📦 Đóng gói Lưu trữ Toàn bộ Minh chứng, Logs, Manifests & Checkpoints
- Nén toàn bộ kết quả, biểu đồ PNG, logs, telemetry và manifest JSON để nghiệm thu khóa luận tốt nghiệp.


In [ ]:
# Cell 10: Đóng gói Lưu trữ Toàn bộ Minh chứng (Logs, Checkpoints, Telemetry, Reports)
import os, shutil, subprocess, json, time

archive_dir = '/kaggle/working/stair5_v4_artifacts'
os.makedirs(archive_dir, exist_ok=True)

# 1. Thu thập logs và telemetry
target_dirs = [
    '/kaggle/working/logs/stair5_v4',
    '/kaggle/working/reports',
    '/kaggle/working/runs/stair5_v4',
]
for src in target_dirs:
    if os.path.exists(src):
        dst = os.path.join(archive_dir, os.path.basename(src))
        shutil.copytree(src, dst, dirs_exist_ok=True)

# 2. Tạo Manifest JSON tổng hợp
manifest = {
    "project": "STAIR5-v4 / NLGCL-CSE",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "datasets": {
        "sports": {
            "test_metrics": metrics_sports if 'metrics_sports' in locals() else {},
            "best_epoch": ep_sports if 'ep_sports' in locals() else None,
            "train_time_min": sports_time / 60 if 'sports_time' in locals() else None,
        },
        "baby": {
            "test_metrics": metrics_baby if 'metrics_baby' in locals() else {},
            "best_epoch": ep_baby if 'ep_baby' in locals() else None,
            "train_time_min": baby_time / 60 if 'baby_time' in locals() else None,
        },
        "electronics": {
            "test_metrics": metrics_elec if 'metrics_elec' in locals() else {},
            "best_epoch": ep_elec if 'ep_elec' in locals() else None,
            "train_time_min": electronics_time / 60 if 'electronics_time' in locals() else None,
        }
    }
}

manifest_path = os.path.join(archive_dir, "stair5_v4_manifest.json")
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# 3. Nén file zip hoàn chỉnh
zip_path = '/kaggle/working/stair5_v4_complete_artifacts.zip'
shutil.make_archive('/kaggle/working/stair5_v4_complete_artifacts', 'zip', archive_dir)

print("=" * 80)
print(f"📦 ĐÓNG GÓI HOÀN TẤT THÀNH CÔNG!")
print(f"  * Thư mục Artifacts : {archive_dir}")
print(f"  * File Nén Tổng Hợp : {zip_path} ({os.path.getsize(zip_path)/(1024**2):.2f} MB)")
print("=" * 80)
